<a href="https://colab.research.google.com/github/lgonperez3-glitch/MLSM-labs/blob/main/Lab2/lab2_task1_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 — Task 1: clasificación de géneros musicales con una CNN

**Versión preparada para Google Colab.** Entrena una CNN desde cero sobre imágenes de espectrogramas, compara 16 configuraciones en dos etapas durante 50 épocas y presenta tablas, gráficas y matrices de confusión.

Antes de ejecutar el entrenamiento, selecciona un runtime GPU en **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** (o la GPU disponible). El dataset `Data/images_original` no se incluye en GitHub; la celda siguiente permite localizarlo en Google Drive.


In [ ]:
# Configuración de Google Colab y localización del dataset
import pandas as pd
from IPython.display import display
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

## 1. Importaciones y configuración general

Importamos las librerías necesarias y fijamos las semillas aleatorias para facilitar la reproducibilidad. Ajusta `DATA_DIR` si las imágenes están en otra carpeta.

In [ ]:
# Si falta alguna librería, ejecuta esta línea en una celda aparte:
# %pip install tensorflow scikit-learn matplotlib pandas

import os
import random
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Semillas aleatorias para facilitar la reproducibilidad.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

DATA_DIR = "/content/drive/MyDrive/MLSM/Data/images_original"
BATCH_SIZE = 32
MAX_EPOCHS = 50

# Mostrar la configuración general como tabla, sin prints sueltos.
display(pd.DataFrame([
    {"Elemento": "TensorFlow", "Valor": tf.__version__},
    {"Elemento": "Épocas por configuración", "Valor": MAX_EPOCHS},
    {"Elemento": "Tamaño de lote", "Valor": BATCH_SIZE},
    {"Elemento": "Semilla aleatoria", "Valor": SEED},
    {"Elemento": "Carpeta de imágenes", "Valor": DATA_DIR},
]))


## 2. Cargar las imágenes y dividir los datos

Cada subcarpeta de `images_original` debe representar un género musical. Primero reservamos el 20 % de las imágenes para test. El 80 % restante se divide en entrenamiento y validación.

La partición final es aproximadamente **64 % entrenamiento, 16 % validación y 20 % test**. La división es estratificada, de modo que cada conjunto conserve una proporción similar de cada género. El conjunto de test no se utiliza para elegir configuraciones.

In [ ]:
# Comprobar que la ruta existe antes de leer los archivos.
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"No encuentro la carpeta '{DATA_DIR}'. Cambia DATA_DIR para indicar la ruta correcta."
    )

# Cada subcarpeta representa un género musical.
class_names = sorted(
    folder for folder in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, folder))
)

valid_extensions = (".png", ".jpg", ".jpeg", ".bmp")
paths = []
labels = []

for label, genre in enumerate(class_names):
    genre_folder = os.path.join(DATA_DIR, genre)
    for filename in sorted(os.listdir(genre_folder)):
        if filename.lower().endswith(valid_extensions):
            paths.append(os.path.join(genre_folder, filename))
            labels.append(label)

paths = np.asarray(paths)
labels = np.asarray(labels)

if len(paths) == 0:
    raise ValueError("No se encontraron imágenes. Comprueba DATA_DIR.")
if len(class_names) < 2:
    raise ValueError("Se necesitan al menos dos carpetas de géneros.")

# División estratificada: 64 % train, 16 % validación y 20 % test.
dev_paths, test_paths, dev_labels, test_labels = train_test_split(
    paths, labels, test_size=0.20, random_state=SEED, stratify=labels
)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    dev_paths, dev_labels, test_size=0.20, random_state=SEED, stratify=dev_labels
)

# Tabla del tamaño de cada partición.
split_df = pd.DataFrame([
    {"Conjunto": "Entrenamiento", "Imágenes": len(train_paths), "Porcentaje": 100 * len(train_paths) / len(paths)},
    {"Conjunto": "Validación", "Imágenes": len(val_paths), "Porcentaje": 100 * len(val_paths) / len(paths)},
    {"Conjunto": "Test", "Imágenes": len(test_paths), "Porcentaje": 100 * len(test_paths) / len(paths)},
    {"Conjunto": "TOTAL", "Imágenes": len(paths), "Porcentaje": 100.0},
])
display(split_df.round(2))

# Tabla con la distribución de géneros en cada partición.
distribution_rows = []
for genre_id, genre in enumerate(class_names):
    distribution_rows.append({
        "Género": genre,
        "Entrenamiento": int(np.sum(train_labels == genre_id)),
        "Validación": int(np.sum(val_labels == genre_id)),
        "Test": int(np.sum(test_labels == genre_id)),
        "Total": int(np.sum(labels == genre_id)),
    })
display(pd.DataFrame(distribution_rows))


## 3. Preparar los datasets de imágenes

Las imágenes se leen desde el disco y se redimensionan al tamaño que necesite cada configuración. Las etiquetas son números enteros, por lo que usaremos la función de pérdida `sparse_categorical_crossentropy`.

Solo barajamos las imágenes de entrenamiento. Validación y test mantienen su orden para que las predicciones puedan compararse con las etiquetas correctas.

In [ ]:
# Leer una imagen, convertirla a RGB y cambiar su resolución.
def load_image(path, label, size):
    image_bytes = tf.io.read_file(path)
    image = tf.image.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, size)
    return image, label


# Crear un dataset por lotes para entrenar o evaluar el modelo.
def make_dataset(file_paths, file_labels, size, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((file_paths, file_labels))

    # Barajar solo entrenamiento; la semilla mantiene el proceso reproducible.
    if training:
        dataset = dataset.shuffle(
            buffer_size=len(file_paths),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    # Cargar y redimensionar imágenes en paralelo.
    dataset = dataset.map(
        lambda path, label: load_image(path, label, size),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # Agrupar imágenes y preparar el siguiente lote mientras se entrena.
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


# Mostrar algunos ejemplos para comprobar que las imágenes y etiquetas se leen bien.
preview_ds = make_dataset(
    train_paths, train_labels, size=(128, 192), training=False
)

for images, labels_batch in preview_ds.take(1):
    plt.figure(figsize=(10, 3))
    for i in range(min(4, len(images))):
        plt.subplot(1, 4, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8))
        plt.title(class_names[int(labels_batch[i].numpy())])
        plt.axis("off")
    plt.tight_layout()
    plt.show()

## 4. Configuraciones experimentales: búsqueda en dos etapas

Se comparan **6 arquitecturas de profundidad** manteniendo los demás hiperparámetros de `BASE0`. Después, se toma la arquitectura con mayor accuracy de validación y se prueban **10 variaciones de un único factor** sobre ella. En total se ejecutan **16 configuraciones**. El test no interviene en la selección.

Los resultados y los mejores pesos de cada ejecución se guardan en Google Drive, para poder reanudar la búsqueda después de una desconexión.

In [ ]:
BASE0 = dict(filters=(32, 64, 128), kernel=(3, 3), dropout=0.3,
             augment=True, lr=1e-3, size=(128, 192))
DEPTH_EXPERIMENTS = {
    "2conv_16-32": dict(filters=(16, 32)),
    "3conv_16-32-64": dict(filters=(16, 32, 64)),
    "3conv_32-64-128": dict(filters=(32, 64, 128)),
    "4conv_16-32-64-128": dict(filters=(16, 32, 64, 128)),
    "4conv_32-64-128-256": dict(filters=(32, 64, 128, 256)),
    "5conv_32-64-128-256-256": dict(filters=(32, 64, 128, 256, 256)),
}
FACTOR_EXPERIMENTS = {
    "dropout_0.0": dict(dropout=0.0),
    "dropout_0.5": dict(dropout=0.5),
    "no_augment": dict(augment=False),
    "lr_3e-4": dict(lr=3e-4),
    "lr_3e-3": dict(lr=3e-3),
    "kernel_5x5": dict(kernel=(5, 5)),
    "kernel_7x3_freq": dict(kernel=(7, 3)),
    "kernel_3x7_time": dict(kernel=(3, 7)),
    "size_64x96": dict(size=(64, 96)),
    "size_192x288": dict(size=(192, 288)),
}
assert len(DEPTH_EXPERIMENTS) == 6 and len(FACTOR_EXPERIMENTS) == 10

def merge_config(base, changes):
    return {**base, **changes}

display(pd.DataFrame([
    {"Configuración": name, "Filtros": str(changes["filters"])}
    for name, changes in DEPTH_EXPERIMENTS.items()
]))
print("La segunda etapa se definirá después de conocer la mejor profundidad.")

## 5. Construir la CNN

La red contiene una serie de bloques convolucionales. Cada bloque aplica una convolución, una activación ReLU y una reducción de resolución con max pooling.

Después usamos global average pooling, una capa densa y dropout opcional antes de la salida. La capa final tiene una neurona por género y utiliza softmax para producir probabilidades.

La normalización de píxeles y el data augmentation se aplican dentro del modelo. Keras activa las capas de augmentation durante el entrenamiento, no durante la evaluación.

In [ ]:
# Crear el modelo a partir de una configuración.
def generate_model(num_classes, config):
    height, width = config["size"]
    inputs = tf.keras.Input(shape=(height, width, 3))
    x = inputs

    # Aumentar ligeramente los datos si la configuración lo solicita.
    # Estas capas solo modifican las imágenes durante el entrenamiento.
    if config["augment"]:
        x = tf.keras.layers.RandomTranslation(0.05, 0.05)(x)
        x = tf.keras.layers.RandomZoom(0.1)(x)
        x = tf.keras.layers.RandomContrast(0.1)(x)

    # Convertir los píxeles de [0, 255] a [0, 1].
    x = tf.keras.layers.Rescaling(1.0 / 255.0)(x)

    # Añadir los bloques convolucionales definidos por la lista de filtros.
    for n_filters in config["filters"]:
        x = tf.keras.layers.Conv2D(
            filters=n_filters,
            kernel_size=config["kernel"],
            padding="same",
            activation="relu"
        )(x)
        x = tf.keras.layers.MaxPooling2D(pool_size=2)(x)

    # Resumir los mapas de características y clasificar el género.
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)

    # Dropout ayuda a reducir el sobreajuste si su valor es mayor que cero.
    if config["dropout"] > 0:
        x = tf.keras.layers.Dropout(config["dropout"])(x)

    outputs = tf.keras.layers.Dense(
        num_classes,
        activation="softmax"
    )(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    # Adam usa el learning rate definido para cada experimento.
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=config["lr"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


# Guardar los pesos de la época con mejor accuracy de validación,
# pero seguir entrenando hasta completar las 50 épocas.
class SaveBestWeights(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.best_val_accuracy = -1.0
        self.best_weights = None
        self.best_epoch = 0

    def on_epoch_end(self, epoch, logs=None):
        val_accuracy = (logs or {}).get("val_accuracy")

        if val_accuracy is not None and val_accuracy > self.best_val_accuracy:
            self.best_val_accuracy = float(val_accuracy)
            self.best_weights = [
                weight.copy() for weight in self.model.get_weights()
            ]
            self.best_epoch = epoch + 1

## 6. Entrenar en dos etapas y guardar resultados

Primero se entrenan las **6 profundidades**. La mejor, según `val_accuracy`, sirve de base para **10 experimentos de factores**. Cada modelo se entrena hasta 50 épocas con `EarlyStopping` (paciencia de 7 épocas, mejora mínima de 0,001). `SaveBestWeights` conserva la época con máxima accuracy de validación.

Tras cada experimento se guardan en Drive un registro CSV, el historial y los pesos de la mejor época. Al volver a ejecutar, se omiten los experimentos cuyos resultados y pesos ya están guardados. **No se utiliza test para elegir el modelo**.

In [ ]:
import json
from pathlib import Path

num_classes = len(class_names)
RESULTS_DIR = Path("/content/drive/MyDrive/MLSM/Lab 2/task1_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = RESULTS_DIR / "task1_two_stage_experiments.csv"
WEIGHTS_DIR = RESULTS_DIR / "weights"
HISTORY_DIR = RESULTS_DIR / "histories"
WEIGHTS_DIR.mkdir(exist_ok=True)
HISTORY_DIR.mkdir(exist_ok=True)

# Cada fila identifica su configuración completa; los pesos e historiales se
# almacenan por separado para permitir reanudar y evaluar sin reentrenar.
def weights_path(name):
    return WEIGHTS_DIR / f"{name}.weights.h5"

def history_path(name):
    return HISTORY_DIR / f"{name}.json"

def csv_rows():
    return pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame()

def run_experiments(experiments, base, group):
    for index, (name, changes) in enumerate(experiments.items(), start=1):
        config = merge_config(base, changes)
        old = csv_rows()
        matching = old[old["name"] == name] if not old.empty else pd.DataFrame()
        if not matching.empty:
            saved = matching.iloc[-1]
            same = (saved["filters"] == "-".join(map(str, config["filters"]))
                    and saved["kernel"] == "x".join(map(str, config["kernel"]))
                    and np.isclose(saved["dropout"], config["dropout"])
                    and str(saved["augment"]).lower() == str(config["augment"]).lower()
                    and np.isclose(saved["lr"], config["lr"])
                    and int(saved["img_h"]) == config["size"][0]
                    and int(saved["img_w"]) == config["size"][1])
            if same and weights_path(name).exists() and history_path(name).exists():
                print(f"[{group} {index}/{len(experiments)}] Ya guardado: {name}")
                continue
            raise RuntimeError(
                f"Existe un resultado previo incompatible o incompleto para {name}. "
                f"Revisa {RESULTS_DIR} antes de continuar."
            )

        print(f"[{group} {index}/{len(experiments)}] Entrenando {name}: {config}")
        tf.keras.backend.clear_session()
        gc.collect()
        tf.keras.utils.set_random_seed(SEED)
        train_ds = make_dataset(train_paths, train_labels, config["size"], training=True)
        val_ds = make_dataset(val_paths, val_labels, config["size"], training=False)
        candidate = generate_model(num_classes, config)
        save_best = SaveBestWeights()
        early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy", mode="max", patience=7,
            min_delta=0.001, restore_best_weights=False
        )
        history = candidate.fit(
            train_ds, validation_data=val_ds, epochs=MAX_EPOCHS,
            shuffle=False, callbacks=[save_best, early_stopping], verbose=1
        )
        best_epoch = save_best.best_epoch
        assert best_epoch > 0 and save_best.best_weights is not None
        row = {
            "name": name, "group": group,
            "filters": "-".join(map(str, config["filters"])),
            "kernel": "x".join(map(str, config["kernel"])),
            "dropout": config["dropout"], "augment": config["augment"],
            "lr": config["lr"], "img_h": config["size"][0], "img_w": config["size"][1],
            "val_accuracy": save_best.best_val_accuracy, "best_epoch": best_epoch,
            "epochs": len(history.history["loss"]),
            "train_accuracy": float(history.history["accuracy"][best_epoch - 1]),
            "train_loss": float(history.history["loss"][best_epoch - 1]),
            "val_loss": float(history.history["val_loss"][best_epoch - 1]),
        }

        # Guardar artefactos primero; la fila CSV marca la ejecución como completa.
        candidate.set_weights(save_best.best_weights)
        candidate.save_weights(str(weights_path(name)))
        with open(history_path(name), "w") as file:
            json.dump({k: [float(v) for v in values]
                       for k, values in history.history.items()}, file)
        pd.DataFrame([row]).to_csv(
            RESULTS_CSV, mode="a", header=not RESULTS_CSV.exists(), index=False
        )
        print(f"Guardado: {name}, val_accuracy={row['val_accuracy']:.4f}")
        del candidate, history, train_ds, val_ds, save_best, early_stopping
        tf.keras.backend.clear_session()
        gc.collect()

# Etapa 1: elegir la mejor arquitectura de profundidad.
run_experiments(DEPTH_EXPERIMENTS, BASE0, "depth")
depth_df = csv_rows().query("group == 'depth'").sort_values(
    ["val_accuracy", "name"], ascending=[False, True]
)
assert len(depth_df) == len(DEPTH_EXPERIMENTS), "Faltan experimentos de profundidad."
best_depth_filters = tuple(map(int, depth_df.iloc[0]["filters"].split("-")))
FACTOR_BASE = merge_config(BASE0, {"filters": best_depth_filters})
print("Mejor profundidad:", best_depth_filters)
print("Base para la segunda etapa:", FACTOR_BASE)

# Etapa 2: variar un único factor respecto a la mejor profundidad.
run_experiments(FACTOR_EXPERIMENTS, FACTOR_BASE, "factor")
all_df = csv_rows()
assert len(all_df) == 16, "Se esperan 6 profundidades y 10 factores."

def config_from_row(row):
    return {
        "filters": tuple(map(int, str(row["filters"]).split("-"))),
        "kernel": tuple(map(int, str(row["kernel"]).split("x"))),
        "dropout": float(row["dropout"]),
        "augment": str(row["augment"]).lower() == "true",
        "lr": float(row["lr"]),
        "size": (int(row["img_h"]), int(row["img_w"])),
    }

experiment_results = {}
for _, row in all_df.iterrows():
    with open(history_path(row["name"])) as file:
        saved_history = json.load(file)
    experiment_results[row["name"]] = {
        "config": config_from_row(row), "history": saved_history,
        "val_accuracy": float(row["val_accuracy"]),
        "best_epoch": int(row["best_epoch"]), "epochs": int(row["epochs"]),
        "train_accuracy": float(row["train_accuracy"]),
        "train_loss": float(row["train_loss"]),
        "val_loss": float(row["val_loss"]),
    }

winner = all_df.sort_values(["val_accuracy", "name"], ascending=[False, True]).iloc[0]
best_name = str(winner["name"])
best_val_accuracy = float(winner["val_accuracy"])
best_config = config_from_row(winner)
best_weights_file = weights_path(best_name)
print(f"Ganador: {best_name}, accuracy de validación: {best_val_accuracy:.4f}")
print(f"CSV: {RESULTS_CSV}")
display(all_df.sort_values("val_accuracy", ascending=False).reset_index(drop=True))


## 7. Tabla comparativa de resultados

Se comparan las **16 configuraciones** (6 de profundidad y 10 de factores), ordenadas por accuracy de validación. Los datos se leen del CSV guardado en Google Drive y los historiales se conservan para visualizar las curvas incluso después de reconectar Colab.

In [ ]:
# Tabla comparativa con el rendimiento de todas las configuraciones.
results_rows = []
for name, run in experiment_results.items():
    config = run["config"]
    results_rows.append({
        "Configuración": name,
        "Accuracy train": run["train_accuracy"],
        "Accuracy validación": run["val_accuracy"],
        "Loss train": run["train_loss"],
        "Loss validación": run["val_loss"],
        "Mejor época": run["best_epoch"],
        "Épocas completadas": run["epochs"],
        "Filtros": " → ".join(map(str, config["filters"])),
        "Kernel": f'{config["kernel"][0]}×{config["kernel"][1]}',
        "Dropout": config["dropout"],
        "Augmentation": "Sí" if config["augment"] else "No",
        "Learning rate": config["lr"],
        "Resolución": f'{config["size"][0]}×{config["size"][1]}',
    })

results_df = pd.DataFrame(results_rows)
results_df = results_df.sort_values(
    "Accuracy validación", ascending=False
).reset_index(drop=True)

display(results_df.round(4))

# Gráfica comparativa de accuracy de entrenamiento y validación.
plot_df = results_df.sort_values("Accuracy validación", ascending=True)
fig, ax = plt.subplots(figsize=(12, 7))
positions = np.arange(len(plot_df))
height = 0.38
ax.barh(positions - height / 2, plot_df["Accuracy train"], height, label="Train")
ax.barh(positions + height / 2, plot_df["Accuracy validación"], height, label="Validación")
ax.set_yticks(positions)
ax.set_yticklabels(plot_df["Configuración"])
ax.set_xlabel("Accuracy")
ax.set_title("Comparación de accuracy por configuración")
ax.set_xlim(0, 1)
ax.grid(axis="x", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

# Gráfica comparativa de loss en entrenamiento y validación.
fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(positions - height / 2, plot_df["Loss train"], height, label="Train")
ax.barh(positions + height / 2, plot_df["Loss validación"], height, label="Validación")
ax.set_yticks(positions)
ax.set_yticklabels(plot_df["Configuración"])
ax.set_xlabel("Loss (menor es mejor)")
ax.set_title("Comparación de loss por configuración")
ax.grid(axis="x", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

# Tabla con la configuración seleccionada según validación.
display(results_df.loc[[0], ["Configuración", "Accuracy train", "Accuracy validación",
                             "Loss train", "Loss validación", "Mejor época",
                             "Épocas completadas", "Filtros", "Kernel", "Dropout",
                             "Augmentation", "Learning rate", "Resolución"]])


## 8. Evaluación final en el conjunto de test

Esta es la única sección que utiliza test para evaluar el rendimiento final. Reconstruimos la configuración ganadora, cargamos sus mejores pesos de validación y calculamos las métricas sobre las imágenes reservadas.

No se deben cambiar los parámetros en función de estos resultados; hacerlo convertiría el test en parte del proceso de selección.

In [ ]:
# Reconstruir la configuración ganadora y cargar sus mejores pesos.
tf.keras.backend.clear_session()
model = generate_model(num_classes, best_config)
model.load_weights(str(best_weights_file))

# El conjunto de test se usa solo para la evaluación final.
test_ds = make_dataset(
    test_paths, test_labels, best_config["size"], training=False
)
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)

# Tabla resumen de evaluación final.
display(pd.DataFrame([{
    "Modelo seleccionado": best_name,
    "Loss en test": test_loss,
    "Accuracy en test": test_accuracy,
    "Imágenes de test": len(test_labels),
}]).round(4))

# Predicciones e informe de clasificación como tabla.
y_prob = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
y_true = test_labels

report = classification_report(
    y_true, y_pred,
    labels=range(num_classes),
    target_names=class_names,
    zero_division=0,
    output_dict=True
)
report_df = pd.DataFrame(report).T
report_df.index.name = "Clase / métrica"
display(report_df.round(4))

# Matriz de confusión con recuentos absolutos.
cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, xticks_rotation=45, colorbar=False, values_format="d"
)
ax.set_title(f"Matriz de confusión (número de imágenes) — {best_name}")
ax.set_xlabel("Género predicho")
ax.set_ylabel("Género real")
plt.tight_layout()
plt.show()

# Matriz normalizada por género real: muestra la proporción de aciertos y errores.
cm_normalized = confusion_matrix(
    y_true, y_pred, labels=range(num_classes), normalize="true"
)
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(cm_normalized, display_labels=class_names).plot(
    ax=ax, xticks_rotation=45, colorbar=True, values_format=".2f"
)
ax.set_title(f"Matriz de confusión normalizada — {best_name}")
ax.set_xlabel("Género predicho")
ax.set_ylabel("Género real")
plt.tight_layout()
plt.show()


## 9. Visualizar las curvas de aprendizaje

Finalmente, mostramos las curvas de accuracy y loss de la configuración seleccionada. Sirven para observar cómo evoluciona el entrenamiento y si aparece una diferencia importante entre entrenamiento y validación.

In [ ]:
# Curvas de aprendizaje del mejor modelo: accuracy.
selected_history = experiment_results[best_name]["history"]
epochs_range = range(1, len(selected_history["accuracy"]) + 1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs_range, selected_history["accuracy"], label="Entrenamiento")
ax.plot(epochs_range, selected_history["val_accuracy"], label="Validación")
ax.set_xlabel("Época")
ax.set_ylabel("Accuracy")
ax.set_title(f"Accuracy por época — {best_name}")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

# Curvas de aprendizaje del mejor modelo: loss.
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs_range, selected_history["loss"], label="Entrenamiento")
ax.plot(epochs_range, selected_history["val_loss"], label="Validación")
ax.set_xlabel("Época")
ax.set_ylabel("Loss")
ax.set_title(f"Loss por época — {best_name}")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

# Tabla por época para consultar los valores exactos del modelo ganador.
history_df = pd.DataFrame({
    "Época": list(epochs_range),
    "Accuracy train": selected_history["accuracy"],
    "Accuracy validación": selected_history["val_accuracy"],
    "Loss train": selected_history["loss"],
    "Loss validación": selected_history["val_loss"],
})
display(history_df.round(4))


## Resumen

- Se ejecutan **6 experimentos de profundidad** y **10 de hiperparámetros** sobre la mejor profundidad: 16 configuraciones.
- Cada experimento tiene un máximo de 50 épocas y utiliza Early Stopping.
- Los resultados, historiales y mejores pesos se guardan tras cada experimento en `MyDrive/MLSM/Lab 2/task1_results/`.
- Si Colab se desconecta, se omiten los experimentos completos al volver a ejecutar la celda de entrenamiento.
- La configuración ganadora se elige por validación y el test solo se utiliza para la evaluación final.

**Antes de ejecutar:** verifica que `DATA_DIR` apunte a `images_original` en tu Drive.